# Notebook 3 — Score de confiance et classification *fit for purpose*

## Position dans le pipeline

`NB1 → NB2 → **NB3 (vous êtes ici)** → NB4`  
Guide : `00_PIPELINE_PAS_A_PAS.md`

## Objectif

Agréger les **preuves QC** en diagnostic de confiance décomposable et classes d’usage (*fit for purpose*).

## Entrées attendues (Notebook 2)

- `data/processed/opensensemap_qc_metrics_station_variable.csv` (**obligatoire**)
- `data/processed/opensensemap_qc_parameters.json`
- (optionnel) observations flaggées — désactivé par défaut (fichier ~2,4 M lignes)

## Sorties attendues

- `data/processed/opensensemap_trust_scores_series.csv`
- `data/processed/opensensemap_trust_scores_stations.csv`
- `data/processed/opensensemap_fit_for_purpose_summary.csv`
- `data/processed/opensensemap_trust_parameters.json`

## Produits

1. scores dimensionnels (0–1) ;
2. score global pondéré ;
3. classification *fit for purpose* ;
4. exports pour le Notebook 4.

> Une anomalie QC n’est pas une preuve d’erreur métrologique.  
> Le score estime une **confiance relative et contextuelle**, pas une exactitude absolue.

Poids / seuils alignés sur `implement_trust_framework.py`.  
Si ce script a déjà tourné, les sorties existent : vous pouvez passer au **Notebook 4**, ou relancer ici pour la sensibilité des poids.


## 1. Bibliothèques et chemins

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

METRICS_CSV = PROCESSED_DIR / "opensensemap_qc_metrics_station_variable.csv"
QC_PARAMS_JSON = PROCESSED_DIR / "opensensemap_qc_parameters.json"
TRUST_SERIES_CSV = PROCESSED_DIR / "opensensemap_trust_scores_series.csv"
TRUST_STATIONS_CSV = PROCESSED_DIR / "opensensemap_trust_scores_stations.csv"
FFP_SUMMARY_CSV = PROCESSED_DIR / "opensensemap_fit_for_purpose_summary.csv"
TRUST_PARAMS_JSON = PROCESSED_DIR / "opensensemap_trust_parameters.json"

# Score observationnel : False par défaut (évite de charger ~2,4 M lignes)
LOAD_OBS_TRUST = False

print("Projet  :", PROJECT_ROOT)
print("Entrées :", PROCESSED_DIR)
print("Rapports:", REPORTS_DIR)
print("Obs trust:", "activé" if LOAD_OBS_TRUST else "désactivé (recommandé)")


## 2. Paramètres du modèle de confiance

Les poids sont des **choix méthodologiques explicites** (à justifier dans le mémoire).  
Ils sont renormalisés sur les dimensions non manquantes (ex. spatial désactivé → poids spatial ignoré).

Une analyse de sensibilité compare ensuite d’autres jeux de poids.


In [ ]:
# Pondérations (alignées sur implement_trust_framework.py)
# Renormalisées automatiquement si une dimension est NaN (ex. spatial off).
TRUST_WEIGHTS = {
    "dim_completeness": 0.20,   # disponibilité temporelle
    "dim_physical": 0.25,       # plausibilité physique
    "dim_stability": 0.20,      # valeurs figées + sauts
    "dim_statistical": 0.15,    # outliers robustes
    "dim_continuity": 0.10,     # interruptions (gaps)
    "dim_metadata": 0.05,       # unités / métadonnées
    "dim_spatial": 0.05,        # cohérence spatiale (si QC spatial activé)
}

# Seuils fit-for-purpose (alignés script ; à justifier dans le mémoire)
FFP_THRESHOLDS = {
    "usage_scientifique_restreint": {
        "trust_min": 0.85,
        "completeness_min": 0.85,
        "physical_min": 0.97,
        "stability_min": 0.90,
        "statistical_min": 0.90,
    },
    "comparaison_relative": {
        "trust_min": 0.70,
        "completeness_min": 0.70,
        "physical_min": 0.90,
        "stability_min": 0.80,
        "statistical_min": 0.80,
    },
    "suivi_tendances": {
        "trust_min": 0.60,
        "completeness_min": 0.60,
        "physical_min": 0.85,
        "stability_min": 0.75,
    },
    "exploratoire": {
        "trust_min": 0.40,
        "completeness_min": 0.30,
        "physical_min": 0.70,
    },
}

# Ordre du plus exigeant au moins exigeant
FFP_ORDER = [
    "usage_scientifique_restreint",
    "comparaison_relative",
    "suivi_tendances",
    "exploratoire",
]

print("Poids :", TRUST_WEIGHTS)
print("Classes FFP :", FFP_ORDER)
print("Somme des poids :", round(sum(TRUST_WEIGHTS.values()), 3))


## 0. Contrôle rapide — scores déjà disponibles ?

Si les fichiers `opensensemap_trust_*` existent déjà, vous pouvez **sauter** ce notebook et ouvrir le **Notebook 4**.  
Relancer NB3 reste utile pour les figures et l’analyse de sensibilité des poids.


In [ ]:
expected_trust = [
    TRUST_SERIES_CSV,
    TRUST_STATIONS_CSV,
    FFP_SUMMARY_CSV,
    TRUST_PARAMS_JSON,
]
present = [p for p in expected_trust if p.exists()]
missing = [p for p in expected_trust if not p.exists()]

print(f"Sorties confiance présentes : {len(present)}/{len(expected_trust)}")
for p in present:
    print("  OK ", p.name)
for p in missing:
    print("  -- ", p.name)

if not missing:
    print("\nScores déjà disponibles → Notebook 4 possible.")
    print("Continuer ici pour recalculer / sensibilité / figures.")
else:
    print("\nExécuter NB3 jusqu'à l'export (ou : python implement_trust_framework.py).")

if not METRICS_CSV.exists():
    print("ATTENTION : métriques QC absentes — exécuter d'abord le Notebook 2.")


## 3. Chargement des sorties du Notebook 2

In [ ]:
def resolve_existing(*candidates):
    for path in candidates:
        if path is not None and Path(path).exists():
            return Path(path)
    return None

metrics_path = resolve_existing(METRICS_CSV)
if metrics_path is None:
    raise FileNotFoundError(
        "Métriques QC introuvables.\n"
        "Exécutez d'abord le Notebook 2 jusqu'à l'export.\n"
        f"Attendu : {METRICS_CSV}"
    )

metrics = pd.read_csv(metrics_path)

params_path = resolve_existing(QC_PARAMS_JSON)
qc_params = {}
if params_path:
    with open(params_path, encoding="utf-8") as f:
        qc_params = json.load(f)

# Observations flaggées : uniquement si LOAD_OBS_TRUST=True
qc_obs = None
qc_obs_path = None
if LOAD_OBS_TRUST:
    qc_obs_path = resolve_existing(
        PROCESSED_DIR / "opensensemap_measurements_qc_flagged.parquet",
        PROCESSED_DIR / "opensensemap_measurements_qc_flagged.csv",
    )
    if qc_obs_path is not None:
        if qc_obs_path.suffix.lower() == ".parquet":
            qc_obs = pd.read_parquet(qc_obs_path)
        else:
            print("Chargement CSV observations (peut être long)...")
            qc_obs = pd.read_csv(qc_obs_path, low_memory=False)

print(f"Métriques chargées : {metrics_path}")
print(f"Dimensions         : {metrics.shape}")
print(f"Paramètres QC      : {params_path if params_path else 'non trouvés'}")
if LOAD_OBS_TRUST:
    print(f"Observations QC    : {qc_obs_path if qc_obs_path else 'non trouvées'}")
else:
    print("Observations QC    : non chargées (LOAD_OBS_TRUST=False)")
display(metrics.head())


## 4. Construction des scores dimensionnels

Transformation retenue : pour un taux d’anomalie $r \in [0,1]$,

$$
d = 1 - r
$$

La complétude est déjà un indicateur positif ($d_{\mathrm{comp}} = c$).

La stabilité combine valeurs figées et sauts :

$$
d_{\mathrm{stab}} = 1 - \frac{r_{\mathrm{stuck}} + r_{\mathrm{jump}}}{2}
$$

In [ ]:
def clip01(series):
    return pd.to_numeric(series, errors="coerce").clip(0.0, 1.0)

trust = metrics.copy()

# Complétude : utiliser completeness_rate si présent, sinon valid_observation_rate
if "completeness_rate" in trust.columns:
    trust["dim_completeness"] = clip01(trust["completeness_rate"])
elif "completeness_percent" in trust.columns:
    trust["dim_completeness"] = clip01(trust["completeness_percent"] / 100.0)
else:
    trust["dim_completeness"] = clip01(trust.get("valid_observation_rate"))

trust["dim_physical"] = 1.0 - clip01(trust.get("physical_anomaly_rate", 0.0))
trust["dim_statistical"] = 1.0 - clip01(trust.get("statistical_anomaly_rate", 0.0))
trust["dim_continuity"] = 1.0 - clip01(trust.get("gap_event_rate", 0.0))
trust["dim_metadata"] = 1.0 - clip01(trust.get("unit_unknown_rate", 0.0))

stuck = clip01(trust.get("stuck_value_rate", 0.0))
jumps = clip01(trust.get("temporal_jump_rate", 0.0))
trust["dim_stability"] = 1.0 - 0.5 * (stuck + jumps)

# Spatial : NaN si QC spatial non exécuté / taux nul et non informatif
spatial_rate = trust.get("spatial_inconsistency_rate")
run_spatial = bool(qc_params.get("run_spatial_qc", False))
if spatial_rate is not None and run_spatial:
    trust["dim_spatial"] = 1.0 - clip01(spatial_rate)
else:
    trust["dim_spatial"] = np.nan

DIM_COLS = [
    "dim_completeness",
    "dim_physical",
    "dim_stability",
    "dim_statistical",
    "dim_continuity",
    "dim_metadata",
    "dim_spatial",
]

display(trust[["station_id", "variable"] + DIM_COLS].head(15))


## 5. Score global de confiance

$$
T = \frac{\sum_i w_i \, d_i}{\sum_i w_i}
$$

où la somme porte uniquement sur les dimensions non manquantes (ex. spatial désactivé).

In [ ]:
def weighted_trust(row, weights):
    num = 0.0
    den = 0.0
    for dim, weight in weights.items():
        value = row.get(dim, np.nan)
        if pd.notna(value):
            num += weight * float(value)
            den += weight
    if den == 0:
        return np.nan
    return num / den

trust["trust_score"] = trust.apply(
    lambda row: weighted_trust(row, TRUST_WEIGHTS),
    axis=1,
)

# Contribution de chaque dimension (pour transparence / explicabilité)
for dim, weight in TRUST_WEIGHTS.items():
    contrib_col = f"contrib_{dim}"
    trust[contrib_col] = np.where(
        trust[dim].notna(),
        weight * trust[dim],
        np.nan,
    )

summary_cols = [
    c for c in [
        "station_id", "station_name", "variable",
        "observations_qc", "completeness_rate",
        *DIM_COLS, "trust_score",
    ] if c in trust.columns
]

display(
    trust[summary_cols]
    .sort_values("trust_score", ascending=True)
    .head(20)
)

print("Score médian :", float(trust["trust_score"].median()))
print("Score moyen  :", float(trust["trust_score"].mean()))


## 6. Classification *fit for purpose*

La classe attribuée est la **plus exigeante** pour laquelle tous les seuils sont respectés.  
Sinon : `non_recommande`.

| Classe | Intention d’usage |
|---|---|
| `usage_scientifique_restreint` | analyses prudentes, avec réserves explicites |
| `comparaison_relative` | comparaisons locales / relatives |
| `suivi_tendances` | tendances temporelles qualitatives |
| `exploratoire` | visualisation, exploration citoyenne |
| `non_recommande` | confiance insuffisante pour un usage structuré |

In [ ]:
FFP_MAP = {
    "trust_min": "trust_score",
    "completeness_min": "dim_completeness",
    "physical_min": "dim_physical",
    "stability_min": "dim_stability",
    "statistical_min": "dim_statistical",
    "metadata_min": "dim_metadata",
    "continuity_min": "dim_continuity",
    "spatial_min": "dim_spatial",
}

def meets_ffp(row, rules):
    """Tous les seuils définis pour la classe doivent être atteints."""
    for rule_key, threshold in rules.items():
        col = FFP_MAP.get(rule_key)
        if col is None:
            continue
        value = row.get(col, np.nan)
        if pd.isna(value):
            return False
        if float(value) < float(threshold):
            return False
    return True

def assign_ffp(row):
    for label in FFP_ORDER:
        if meets_ffp(row, FFP_THRESHOLDS[label]):
            return label
    return "non_recommande"

trust["fit_for_purpose"] = trust.apply(assign_ffp, axis=1)

ffp_counts = (
    trust["fit_for_purpose"]
    .value_counts(dropna=False)
    .rename_axis("classe")
    .reset_index(name="n_series")
)
ffp_counts["proportion"] = ffp_counts["n_series"] / len(trust)

display(ffp_counts)
display(
    trust[
        ["station_id", "station_name", "variable", "trust_score", "fit_for_purpose"]
    ]
    .sort_values(["fit_for_purpose", "trust_score"])
    .head(30)
)


## 7. Agrégation au niveau station

Une station possède plusieurs variables. On agrège par médiane des scores,  
et on retient la classe FFP **la plus restrictive** observée parmi ses variables.

In [ ]:
FFP_RANK = {
    "non_recommande": 0,
    "exploratoire": 1,
    "suivi_tendances": 2,
    "comparaison_relative": 3,
    "usage_scientifique_restreint": 4,
}
RANK_TO_FFP = {v: k for k, v in FFP_RANK.items()}

def most_restrictive_ffp(series):
    ranks = [FFP_RANK.get(x, 0) for x in series.dropna()]
    if not ranks:
        return "non_recommande"
    return RANK_TO_FFP[min(ranks)]

station_trust = (
    trust.groupby(["station_id"], dropna=False)
    .agg(
        station_name=("station_name", "first"),
        n_variables=("variable", "nunique"),
        trust_score_median=("trust_score", "median"),
        trust_score_min=("trust_score", "min"),
        dim_completeness_median=("dim_completeness", "median"),
        dim_physical_median=("dim_physical", "median"),
        dim_stability_median=("dim_stability", "median"),
        fit_for_purpose_station=("fit_for_purpose", most_restrictive_ffp),
    )
    .reset_index()
    .sort_values("trust_score_median", ascending=False)
)

display(station_trust.head(20))
display(
    station_trust["fit_for_purpose_station"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="n_stations")
)


## 8. Score observationnel (optionnel)

Activé seulement si `LOAD_OBS_TRUST = True` dans la cellule de configuration  
(désactivé par défaut : évite de recharger ~2,4 M lignes).

Si activé et que le fichier d’observations flaggées est disponible, on construit un score  
par observation à partir des drapeaux majeurs/mineurs du Notebook 2.

In [ ]:
obs_trust = None

if qc_obs is not None and not qc_obs.empty:
    obs = qc_obs.copy()

    major_flags = [
        "flag_duplicate_timestamp",
        "flag_physical_implausibility",
        "flag_stuck_value",
        "flag_temporal_jump",
        "flag_statistical_anomaly",
        "flag_spatial_inconsistency",
    ]
    minor_flags = [
        "flag_unit_unknown",
        "flag_gap_before",
    ]

    for col in major_flags + minor_flags:
        if col not in obs.columns:
            obs[col] = False
        obs[col] = obs[col].fillna(False).astype(bool)

    # Pénalités additives bornées : score ∈ [0, 1]
    major_penalty = 0.15
    minor_penalty = 0.05

    obs["obs_trust_score"] = (
        1.0
        - major_penalty * obs[major_flags].sum(axis=1)
        - minor_penalty * obs[minor_flags].sum(axis=1)
    ).clip(0.0, 1.0)

    # Fusion du score de série pour contextualiser l'observation
    series_key_cols = [c for c in ["station_id", "variable"] if c in obs.columns]
    obs = obs.merge(
        trust[series_key_cols + ["trust_score", "fit_for_purpose"]],
        on=series_key_cols,
        how="left",
        suffixes=("", "_series"),
    )

    obs["obs_trust_combined"] = (
        0.5 * obs["obs_trust_score"] + 0.5 * obs["trust_score"]
    )

    obs_trust = obs
    display(
        obs_trust[
            [
                c for c in [
                    "station_id", "variable", "timestamp", "value_std",
                    "qc_status", "obs_trust_score", "trust_score",
                    "obs_trust_combined", "fit_for_purpose",
                ] if c in obs_trust.columns
            ]
        ].head(15)
    )
else:
    print(
        "Fichier d'observations QC absent : "
        "seuls les scores série/station sont calculés."
    )


## 9. Visualisations diagnostiques

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(trust["trust_score"].dropna(), bins=20, edgecolor="black")
axes[0].set_title("Distribution des scores de confiance (séries)")
axes[0].set_xlabel("trust_score")
axes[0].set_ylabel("Nombre de séries")

ffp_order_plot = FFP_ORDER + ["non_recommande"]
counts = (
    trust["fit_for_purpose"]
    .value_counts()
    .reindex(ffp_order_plot)
    .fillna(0)
)
axes[1].barh(counts.index.astype(str), counts.values)
axes[1].set_title("Classes fit-for-purpose")
axes[1].set_xlabel("Nombre de séries")

plt.tight_layout()
fig_path = REPORTS_DIR / "trust_score_distribution.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée :", fig_path)

dim_means = trust[DIM_COLS].mean(numeric_only=True)
fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.bar(dim_means.index, dim_means.values)
ax2.set_ylim(0, 1)
ax2.set_title("Score moyen par dimension")
ax2.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig2_path = REPORTS_DIR / "trust_dimensions_mean.png"
fig2.savefig(fig2_path, dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée :", fig2_path)


## 10. Analyse de sensibilité simple des poids

On compare le classement obtenu avec des poids égaux vs une variante  
« physique prioritaire » pour vérifier la robustesse qualitative.

In [ ]:
WEIGHTS_EQUAL = {k: 1.0 for k in TRUST_WEIGHTS}

WEIGHTS_PHYSICAL_PRIORITY = {
    "dim_completeness": 0.15,
    "dim_physical": 0.40,
    "dim_stability": 0.20,
    "dim_statistical": 0.10,
    "dim_continuity": 0.05,
    "dim_metadata": 0.05,
    "dim_spatial": 0.05,
}

trust["trust_equal"] = trust.apply(
    lambda r: weighted_trust(r, WEIGHTS_EQUAL), axis=1
)
trust["trust_physical_priority"] = trust.apply(
    lambda r: weighted_trust(r, WEIGHTS_PHYSICAL_PRIORITY), axis=1
)

corr = trust[["trust_score", "trust_equal", "trust_physical_priority"]].corr(
    method="spearman"
)
display(corr)

rank_shift = (
    trust["trust_score"].rank(ascending=False)
    - trust["trust_physical_priority"].rank(ascending=False)
).abs()
print(
    "Décalage médian de rang (baseline vs physique prioritaire) :",
    float(rank_shift.median()),
)


## 11. Export des résultats

In [ ]:
series_out = TRUST_SERIES_CSV
station_out = TRUST_STATIONS_CSV
ffp_out = FFP_SUMMARY_CSV
params_out = TRUST_PARAMS_JSON

trust.to_csv(series_out, index=False)
station_trust.to_csv(station_out, index=False)
ffp_counts.to_csv(ffp_out, index=False)

export_params = {
    "trust_weights": TRUST_WEIGHTS,
    "ffp_thresholds": FFP_THRESHOLDS,
    "ffp_order": FFP_ORDER,
    "aligned_with": "implement_trust_framework.py (partie confiance / FFP)",
    "notes": [
        "Score de confiance relatif et contextuel, non métrologique.",
        "Dimensions manquantes exclues de la renormalisation des poids.",
        "Classe station = classe la plus restrictive parmi les variables.",
    ],
    "source_metrics": str(metrics_path),
    "source_qc_params": str(params_path) if params_path else None,
    "load_obs_trust": LOAD_OBS_TRUST,
}

with open(params_out, "w", encoding="utf-8") as f:
    json.dump(export_params, f, ensure_ascii=False, indent=2)

if obs_trust is not None:
    obs_out = PROCESSED_DIR / "opensensemap_trust_scores_observations.csv"
    keep_cols = [
        c for c in [
            "station_id", "station_name", "variable", "sensor_id",
            "timestamp", "value_std", "unit_std", "qc_status",
            "obs_trust_score", "trust_score", "obs_trust_combined",
            "fit_for_purpose",
        ] if c in obs_trust.columns
    ]
    obs_trust[keep_cols].to_csv(obs_out, index=False)
    print("Observations :", obs_out)

print("Exports réalisés :")
print("-", series_out)
print("-", station_out)
print("-", ffp_out)
print("-", params_out)
print(f"\nSéries : {len(trust)} | score médian : {trust['trust_score'].median():.3f}")
print("Étape suivante → Notebook 4 (tableaux / figures mémoire).")


## 12. Bilan

Ce notebook transforme les métriques QC en diagnostic de confiance :

- **décomposable** (dimensions explicites) ;
- **reproductible** (poids et seuils exportés en JSON) ;
- **orienté usage** (classes *fit for purpose*) ;
- **non métrologique** (pas de certification d’exactitude absolue).

### Étape suivante

Exécuter le **Notebook 4** (`Notebook_4_Resultats_memoire.ipynb`) pour produire tableaux et figures du mémoire.

Dans la rédaction : justifier les poids (`TRUST_WEIGHTS`) et seuils (`FFP_THRESHOLDS`) comme choix méthodologiques, avec analyse de sensibilité.
